# 11.1. LightFM Hyperparameter Tuning

Цели:
- подбор гиперпараметров по validation
- выбор лучшей конфигурации по `Recall@10`
- оценка лучшей модели на test
- сравнение с baseline из `10_lightfm_item_features.ipynb`
- сохранение `best_config` и `top10_configs`



> CPU profile: i9, используем максимум доступных потоков.



In [1]:
# 1. Импорты и настройки
import ast
import json
import os
import time
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import csr_matrix, hstack, identity
from lightfm import LightFM

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

project_root = Path.cwd()
if not (project_root / 'configs').exists() and (project_root.parent / 'configs').exists():
    project_root = project_root.parent

CONFIG_PATH = project_root / 'configs' / 'base.json'
artifacts_root_dir = 'artifacts'
if CONFIG_PATH.exists():
    with CONFIG_PATH.open('r', encoding='utf-8') as f:
        cfg = json.load(f)
    artifacts_root_dir = (
        cfg.get('paths', {}).get('artifacts_root_dir')
        or cfg.get('data', {}).get('artifacts_root_dir')
        or 'artifacts'
    )

ARTIFACTS_DIR = project_root / artifacts_root_dir
PROCESSED_DIR = ARTIFACTS_DIR / 'processed'
SPLITS_DIR = ARTIFACTS_DIR / 'splits'
EVAL_DIR = SPLITS_DIR / 'eval'

TUNING_DIR = ARTIFACTS_DIR / 'experiments' / 'lightfm_tuning'
CHECKPOINT_DIR = TUNING_DIR / 'checkpoints'
MODELS_DIR = ARTIFACTS_DIR / 'models' / 'lightfm_item_features_tuned'
PRED_DIR = ARTIFACTS_DIR / 'predictions' / 'lightfm_item_features_tuned'
METRICS_DIR = ARTIFACTS_DIR / 'metrics' / 'lightfm_item_features_tuned'

for p in [TUNING_DIR, CHECKPOINT_DIR, MODELS_DIR, PRED_DIR, METRICS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# CPU tuning for i9: используем все доступные потоки
os.environ['OMP_NUM_THREADS'] = str(N_CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(N_CPU_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_CPU_THREADS)
os.environ['NUMEXPR_NUM_THREADS'] = str(N_CPU_THREADS)

K_VALUES = [5, 10, 20, 50]
K_OPT = 10
K_MAX = max(K_VALUES)

MAX_TUNING_RUNS = 36
N_CPU_THREADS = os.cpu_count() or 1

print('project_root =', project_root)
print('ARTIFACTS_DIR =', ARTIFACTS_DIR)


print('CPU threads:', N_CPU_THREADS)




project_root = /Users/maybeabakarov/Desktop/РекомендацииПроект
ARTIFACTS_DIR = /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts


/Users/maybeabakarov/Desktop/РекомендацииПроект/.venv311/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


## 2. Загрузка артефактов


In [2]:
def load_any(paths):
    for p in paths:
        if p.exists():
            if p.suffix == '.csv':
                return pd.read_csv(p), p
            if p.suffix == '.parquet':
                return pd.read_parquet(p), p
            if p.suffix == '.json':
                with p.open('r', encoding='utf-8') as f:
                    return json.load(f), p
    return None, None

artifacts_map = {
    'mf_train_implicit': [
        PROCESSED_DIR / 'mf_train_implicit.csv',
        SPLITS_DIR / 'train_interactions_implicit.csv',
    ],
    'user_mapping': [PROCESSED_DIR / 'user_mapping.csv'],
    'item_mapping': [PROCESSED_DIR / 'item_mapping.csv'],
    'seen_items': [EVAL_DIR / 'seen_items_by_user.csv'],
    'val_ground_truth': [EVAL_DIR / 'val_ground_truth.csv'],
    'test_ground_truth': [EVAL_DIR / 'test_ground_truth.csv'],
    'eval_users_val': [EVAL_DIR / 'eval_users_val.csv'],
    'eval_users_test': [EVAL_DIR / 'eval_users_test.csv'],
    'split_metadata': [SPLITS_DIR / 'split_metadata.json'],
    'item_metadata': [PROCESSED_DIR / 'item_metadata_cleaned.csv'],
    'item_tag_stats': [PROCESSED_DIR / 'item_tag_stats.csv'],
    'candidate_items': [PROCESSED_DIR / 'candidate_items_train.csv'],
    'train_item_popularity': [PROCESSED_DIR / 'train_item_popularity.csv'],
}

loaded = {}
for name, paths in artifacts_map.items():
    obj, path = load_any(paths)
    loaded[name] = {'obj': obj, 'path': path}

required = [
    'mf_train_implicit',
    'seen_items',
    'val_ground_truth',
    'test_ground_truth',
    'eval_users_val',
    'eval_users_test',
    'split_metadata',
    'item_metadata',
    'train_item_popularity',
]
for k in required:
    if loaded[k]['obj'] is None:
        raise FileNotFoundError(f"Не найден обязательный артефакт '{k}'. Проверенные пути: {artifacts_map[k]}")

train_df = loaded['mf_train_implicit']['obj']
user_map = loaded['user_mapping']['obj']
item_map = loaded['item_mapping']['obj']
seen_items = loaded['seen_items']['obj']
val_gt = loaded['val_ground_truth']['obj']
test_gt = loaded['test_ground_truth']['obj']
eval_users_val = loaded['eval_users_val']['obj']
eval_users_test = loaded['eval_users_test']['obj']
split_meta = loaded['split_metadata']['obj']
item_meta = loaded['item_metadata']['obj']
item_tag_stats = loaded['item_tag_stats']['obj']
candidate_items_df = loaded['candidate_items']['obj']
item_popularity = loaded['train_item_popularity']['obj']

if candidate_items_df is None:
    candidate_items_df = pd.DataFrame({'movieId': sorted(pd.to_numeric(train_df['movieId'], errors='coerce').dropna().astype('int64').unique())})

for df in [train_df, user_map, item_map, seen_items, val_gt, test_gt, eval_users_val, eval_users_test, item_meta, item_popularity, candidate_items_df]:
    if df is None:
        continue
    if 'userId' in df.columns:
        df['userId'] = pd.to_numeric(df['userId'], errors='coerce')
    if 'movieId' in df.columns:
        df['movieId'] = pd.to_numeric(df['movieId'], errors='coerce')

for df in [train_df , seen_items, val_gt, test_gt, eval_users_val, eval_users_test, item_meta, item_popularity, candidate_items_df]:
    if 'userId' in df.columns:
        df.dropna(subset=['userId'], inplace=True)
        df['userId'] = df['userId'].astype('int64')
    if 'movieId' in df.columns:
        df.dropna(subset=['movieId'], inplace=True)
        df['movieId'] = df['movieId'].astype('int64')

print('Используемые пути:')
for name in artifacts_map:
    print(f"- {name}: {loaded[name]['path']}")

display(pd.DataFrame([
    {'table': 'mf_train_implicit', 'rows': len(train_df)},
    {'table': 'val_ground_truth', 'rows': len(val_gt)},
    {'table': 'test_ground_truth', 'rows': len(test_gt)},
    {'table': 'eval_users_val', 'rows': len(eval_users_val)},
    {'table': 'eval_users_test', 'rows': len(eval_users_test)},
    {'table': 'seen_items_by_user', 'rows': len(seen_items)},
    {'table': 'item_metadata_cleaned', 'rows': len(item_meta)},
    {'table': 'train_item_popularity', 'rows': len(item_popularity)},
    {'table': 'candidate_items', 'rows': len(candidate_items_df)},
]))




Используемые пути:
- mf_train_implicit: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/processed/mf_train_implicit.csv
- user_mapping: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/processed/user_mapping.csv
- item_mapping: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/processed/item_mapping.csv
- seen_items: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/splits/eval/seen_items_by_user.csv
- val_ground_truth: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/splits/eval/val_ground_truth.csv
- test_ground_truth: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/splits/eval/test_ground_truth.csv
- eval_users_val: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/splits/eval/eval_users_val.csv
- eval_users_test: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/splits/eval/eval_users_test.csv
- split_metadata: /Users/maybeabakarov/Desktop/РекомендацииПроект/artifacts/splits/split_metadata.json
- item_metadata: /

,table,rows
0,mf_train_implicit,36802
1,val_ground_truth,225
2,test_ground_truth,247
3,eval_users_val,18
4,eval_users_test,16
5,seen_items_by_user,36802
6,item_metadata_cleaned,9742
7,train_item_popularity,3484
8,candidate_items,3484


## 3. Подготовка матриц interactions и item features


In [3]:

users = pd.DataFrame({'userId': sorted(train_df['userId'].unique())})
users['user_idx'] = np.arange(len(users), dtype=np.int32)

items = pd.DataFrame({'movieId': sorted(train_df['movieId'].unique())})
items['item_idx'] = np.arange(len(items), dtype=np.int32)

user_to_idx = dict(zip(users['userId'], users['user_idx']))
item_to_idx = dict(zip(items['movieId'], items['item_idx']))
idx_to_item = dict(zip(items['item_idx'], items['movieId']))

train_indexed = (
    train_df[['userId', 'movieId']]
    .drop_duplicates()
    .merge(users, on='userId', how='inner')
    .merge(items, on='movieId', how='inner')
)

interactions = csr_matrix(
    (
        np.ones(len(train_indexed), dtype=np.float32),
        (train_indexed['user_idx'].to_numpy(), train_indexed['item_idx'].to_numpy())
    ),
    shape=(len(users), len(items)),
    dtype=np.float32,
)


def parse_genres(row):
    gl = row.get('genres_list', None)
    if isinstance(gl, list):
        return [str(x).strip() for x in gl if str(x).strip()]
    if isinstance(gl, str) and gl.strip().startswith('['):
        try:
            arr = ast.literal_eval(gl)
            if isinstance(arr, list):
                return [str(x).strip() for x in arr if str(x).strip()]
        except Exception:
            pass
    g = row.get('genres', None)
    if isinstance(g, str):
        return [x.strip() for x in g.split('|') if x.strip() and x.strip() != '(no genres listed)']
    return []

item_meta_local = items[['movieId']].merge(item_meta, on='movieId', how='left')
item_meta_local['genres_parsed'] = item_meta_local.apply(parse_genres, axis=1)

all_genres = sorted({g.lower() for gs in item_meta_local['genres_parsed'] for g in gs})
genre_to_idx = {g: i for i, g in enumerate(all_genres)}

rows, cols, vals = [], [], []
for r, gs in enumerate(item_meta_local['genres_parsed']):
    for g in gs:
        gi = genre_to_idx.get(g.lower())
        if gi is None:
            continue
        rows.append(r)
        cols.append(gi)
        vals.append(1.0)

if len(all_genres) > 0 and len(rows) > 0:
    genre_features = csr_matrix((vals, (rows, cols)), shape=(len(items), len(all_genres)), dtype=np.float32)
    item_features = hstack([identity(len(items), format='csr', dtype=np.float32), genre_features], format='csr')
else:
    item_features = identity(len(items), format='csr', dtype=np.float32)

user_seen = seen_items.groupby('userId')['movieId'].apply(set).to_dict()
pop_ranked = item_popularity.sort_values(['popularity_rank', 'movieId']) if 'popularity_rank' in item_popularity.columns else item_popularity.sort_values(['positive_interactions', 'movieId'], ascending=[False, True])

print('interactions shape:', interactions.shape, 'nnz=', interactions.nnz)
print('item_features shape:', item_features.shape)
print('users/items:', len(users), len(items))




interactions shape: (506, 3484) nnz= 36802
item_features shape: (3484, 3503)
users/items: 506 3484


## 4. Метрики и инференс-функции


In [4]:
def recommend_for_user(model, user_id: int, k: int):
    uidx = user_to_idx.get(int(user_id))
    seen = user_seen.get(int(user_id), set())

    # fallback для неизвестных пользователей
    if uidx is None:
        recs = []
        for mid in pop_ranked['movieId'].tolist():
            if mid in seen:
                continue
            recs.append((int(mid), 0.0))
            if len(recs) >= k:
                break
        return recs

    scores = model.predict(
        user_ids=np.repeat(np.int32(uidx), len(items)),
        item_ids=np.arange(len(items), dtype=np.int32),
        item_features=item_features,
        num_threads=1,
    )
    order = np.argsort(scores)[::-1]

    recs = []
    for idx in order:
        mid = idx_to_item[int(idx)]
        if mid in seen:
            continue
        recs.append((int(mid), float(scores[idx])))
        if len(recs) >= k:
            break
    return recs


def build_predictions(model, eval_users_df, split_name, k):
    rows = []
    users_list = eval_users_df['userId'].drop_duplicates().astype('int64').tolist()
    for uid in users_list:
        recs = recommend_for_user(model, int(uid), k)
        for rank, (mid, score) in enumerate(recs, start=1):
            rows.append({'userId': int(uid), 'rank': rank, 'movieId': int(mid), 'score': float(score), 'split': split_name})
    return pd.DataFrame(rows)


def build_gt_map(gt_df):
    return gt_df.groupby('userId')['movieId'].apply(set).to_dict()


def dcg_at_k(recommended, relevant_set, k):
    score = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant_set:
            score += 1.0 / np.log2(i + 1)
    return score


def evaluate_split(pred_df, gt_df, eval_users_df, candidate_items_df, k_values):
    gt_map = build_gt_map(gt_df)
    eval_users = eval_users_df['userId'].drop_duplicates().astype('int64').tolist()

    rows = []
    for k in k_values:
        recalls, precisions, maps, ndcgs, hitrates = [], [], [], [], []
        all_recommended_items = set()
        list_lengths = []

        for uid in eval_users:
            user_pred = pred_df[(pred_df['userId'] == uid) & (pred_df['rank'] <= k)].sort_values('rank')
            rec_items = user_pred['movieId'].astype('int64').tolist()
            rel_items = gt_map.get(uid, set())

            all_recommended_items.update(rec_items)
            list_lengths.append(len(rec_items))

            if len(rel_items) == 0:
                continue

            hits = [1 if x in rel_items else 0 for x in rec_items]
            n_hits = int(sum(hits))

            recall = n_hits / len(rel_items)
            precision = n_hits / max(k, 1)
            hitrate = 1.0 if n_hits > 0 else 0.0

            cum_hits = 0
            ap_sum = 0.0
            for i, h in enumerate(hits, start=1):
                if h:
                    cum_hits += 1
                    ap_sum += cum_hits / i
            ap = ap_sum / max(1, len(rel_items))

            dcg = dcg_at_k(rec_items, rel_items, k)
            ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(rel_items), k) + 1))
            ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0.0

            recalls.append(recall)
            precisions.append(precision)
            maps.append(ap)
            ndcgs.append(ndcg)
            hitrates.append(hitrate)

        rows.append({
            'K': k,
            'Recall@K': float(np.mean(recalls)) if recalls else 0.0,
            'Precision@K': float(np.mean(precisions)) if precisions else 0.0,
            'MAP@K': float(np.mean(maps)) if maps else 0.0,
            'NDCG@K': float(np.mean(ndcgs)) if ndcgs else 0.0,
            'HitRate@K': float(np.mean(hitrates)) if hitrates else 0.0,
            'Catalog Coverage@K': len(all_recommended_items) / max(1, len(candidate_items_df)),
            'average_list_length': float(np.mean(list_lengths)) if list_lengths else 0.0,
            'full_topk_rate': float(np.mean([1 if x >= k else 0 for x in list_lengths])) if list_lengths else 0.0,
            'unique_recommended_items': int(len(all_recommended_items)),
            'candidate_items': int(len(candidate_items_df)),
        })

    return pd.DataFrame(rows)

candidate_items = candidate_items_df[['movieId']].drop_duplicates().copy()




## 5. Random Search по гиперпараметрам


In [5]:
param_space = {
    'loss': ['warp', 'bpr'],
    'no_components': [32, 64, 96],
    'learning_rate': [0.03, 0.05, 0.08],
    'item_alpha': [1e-6, 1e-5, 1e-4],
    'user_alpha': [1e-6, 1e-5, 1e-4],
    'epochs': [10, 20, 30],
}

N_RANDOM_TRIALS = MAX_TUNING_RUNS
rng = np.random.default_rng(RANDOM_SEED)

configs = []
seen = set()
max_attempts = N_RANDOM_TRIALS * 20
attempts = 0

while len(configs) < N_RANDOM_TRIALS and attempts < max_attempts:
    attempts += 1
    cfg = {
        'loss': str(rng.choice(param_space['loss'])),
        'no_components': int(rng.choice(param_space['no_components'])),
        'learning_rate': float(rng.choice(param_space['learning_rate'])),
        'item_alpha': float(rng.choice(param_space['item_alpha'])),
        'user_alpha': float(rng.choice(param_space['user_alpha'])),
        'epochs': int(rng.choice(param_space['epochs'])),
    }
    key = (
        cfg['loss'],
        cfg['no_components'],
        cfg['learning_rate'],
        cfg['item_alpha'],
        cfg['user_alpha'],
        cfg['epochs'],
    )
    if key in seen:
        continue
    seen.add(key)
    configs.append(cfg)

configs_df = pd.DataFrame(configs)
print('Режим:', 'random search')
print('Запрошено trials:', N_RANDOM_TRIALS)
print('Собрано уникальных конфигураций:', len(configs_df))
display(configs_df.head(20))




Режим: random search
Запрошено trials: 36
Собрано уникальных конфигураций: 36


,loss,no_components,learning_rate,item_alpha,user_alpha,epochs
0,warp,96,0.05,0.000010,0.000010,30
1,warp,96,0.03,0.000001,0.000010,30
2,bpr,96,0.08,0.000100,0.000010,10
3,bpr,64,0.05,0.000010,0.000001,30
4,bpr,64,0.05,0.000100,0.000010,20
5,warp,32,0.03,0.000010,0.000100,10
6,bpr,96,0.03,0.000010,0.000001,30
7,bpr,64,0.03,0.000100,0.000010,30
8,bpr,96,0.08,0.000001,0.000010,20
9,warp,32,0.05,0.000001,0.000100,30


## 6. Тюнинг на validation


In [6]:
val_rows = []
best_model = None
best_cfg = None
best_score = -np.inf  # objective: Recall@10

partial_csv_path = TUNING_DIR / 'lightfm_tuning_validation_results_partial.csv'
partial_jsonl_path = TUNING_DIR / 'lightfm_tuning_runs.jsonl'
best_so_far_cfg_path = CHECKPOINT_DIR / 'best_config_so_far.json'

for i, cfg in enumerate(configs, start=1):
    print(f"[{i}/{len(configs)}] cfg={cfg}")
    t0 = time.perf_counter()

    model = LightFM(
        loss=cfg['loss'],
        no_components=int(cfg['no_components']),
        learning_rate=float(cfg['learning_rate']),
        item_alpha=float(cfg['item_alpha']),
        user_alpha=float(cfg['user_alpha']),
        random_state=RANDOM_SEED,
    )
    model.fit(
        interactions,
        item_features=item_features,
        epochs=int(cfg['epochs']),
        num_threads=N_CPU_THREADS,
        verbose=False,
    )

    fit_time_sec = time.perf_counter() - t0

    val_pred = build_predictions(model, eval_users_val, 'validation', K_MAX)
    val_metrics = evaluate_split(val_pred, val_gt, eval_users_val, candidate_items, K_VALUES)

    row10 = val_metrics[val_metrics['K'] == K_OPT].iloc[0]
    row = {
        **cfg,
        'run_idx': i,
        'fit_time_sec': round(fit_time_sec, 3),
        'Recall@10': float(row10['Recall@K']),
        'Precision@10': float(row10['Precision@K']),
        'MAP@10': float(row10['MAP@K']),
        'NDCG@10': float(row10['NDCG@K']),
        'HitRate@10': float(row10['HitRate@K']),
        'CatalogCoverage@10': float(row10['Catalog Coverage@K']),
    }
    val_rows.append(row)

    # 1) промежуточное сохранение результатов после КАЖДОЙ модели
    partial_df = pd.DataFrame(val_rows).sort_values(['Recall@10', 'NDCG@10', 'MAP@10'], ascending=False).reset_index(drop=True)
    partial_df.to_csv(partial_csv_path, index=False)

    with partial_jsonl_path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

    # 2) вывод конфигурации и метрик текущего прогона
    print(
        f"  -> run={i}, Recall@10={row['Recall@10']:.6f}, NDCG@10={row['NDCG@10']:.6f}, "
        f"MAP@10={row['MAP@10']:.6f}, fit_time_sec={row['fit_time_sec']}"
    )

    # 3) best-so-far checkpoint
    if (row['Recall@10'] > best_score) or (
        np.isclose(row['Recall@10'], best_score) and row['NDCG@10'] > (0 if best_cfg is None else best_cfg.get('NDCG@10', 0))
    ):
        best_score = row['Recall@10']
        best_model = model
        best_cfg = row.copy()

        with best_so_far_cfg_path.open('w', encoding='utf-8') as f:
            json.dump(best_cfg, f, ensure_ascii=False, indent=2)

        np.save(CHECKPOINT_DIR / 'best_user_embeddings_so_far.npy', model.user_embeddings)
        np.save(CHECKPOINT_DIR / 'best_item_embeddings_so_far.npy', model.item_embeddings)
        np.save(CHECKPOINT_DIR / 'best_item_biases_so_far.npy', model.item_biases)
        np.save(CHECKPOINT_DIR / 'best_user_biases_so_far.npy', model.user_biases)

results_val = pd.DataFrame(val_rows).sort_values(['Recall@10', 'NDCG@10', 'MAP@10'], ascending=False).reset_index(drop=True)
display(results_val.head(20))




[1/36] cfg={'loss': 'warp', 'no_components': 96, 'learning_rate': 0.05, 'item_alpha': 1e-05, 'user_alpha': 1e-05, 'epochs': 30}
  -> run=1, Recall@10=0.025512, NDCG@10=0.041290, MAP@10=0.008604, fit_time_sec=6.046
  -> best-so-far обновлён: Recall@10=0.025512
[2/36] cfg={'loss': 'warp', 'no_components': 96, 'learning_rate': 0.03, 'item_alpha': 1e-06, 'user_alpha': 1e-05, 'epochs': 30}
  -> run=2, Recall@10=0.039352, NDCG@10=0.046240, MAP@10=0.014937, fit_time_sec=7.49
  -> best-so-far обновлён: Recall@10=0.039352
[3/36] cfg={'loss': 'bpr', 'no_components': 96, 'learning_rate': 0.08, 'item_alpha': 0.0001, 'user_alpha': 1e-05, 'epochs': 10}
  -> run=3, Recall@10=0.041777, NDCG@10=0.039415, MAP@10=0.010569, fit_time_sec=3.606
  -> best-so-far обновлён: Recall@10=0.041777
[4/36] cfg={'loss': 'bpr', 'no_components': 64, 'learning_rate': 0.05, 'item_alpha': 1e-05, 'user_alpha': 1e-06, 'epochs': 30}
  -> run=4, Recall@10=0.042394, NDCG@10=0.046729, MAP@10=0.012428, fit_time_sec=7.654
  -> bes

,loss,no_components,learning_rate,item_alpha,user_alpha,epochs,run_idx,fit_time_sec,Recall@10,Precision@10,MAP@10,NDCG@10,HitRate@10,CatalogCoverage@10
0,warp,32,0.05,0.000100,0.000001,20,36,1.928,0.052315,0.044444,0.014328,0.051355,0.333333,0.039610
1,bpr,96,0.03,0.000010,0.000001,30,7,11.132,0.048721,0.044444,0.017389,0.054343,0.222222,0.041619
2,warp,96,0.03,0.000100,0.000010,20,19,5.275,0.048611,0.044444,0.012002,0.043677,0.222222,0.039036
3,warp,64,0.03,0.000001,0.000100,20,32,3.874,0.048225,0.044444,0.018770,0.050249,0.222222,0.039610
4,bpr,96,0.08,0.000001,0.000010,20,9,7.317,0.043474,0.038889,0.010202,0.041415,0.277778,0.044776
5,bpr,64,0.05,0.000010,0.000001,30,4,7.654,0.042394,0.038889,0.012428,0.046729,0.277778,0.041332
6,bpr,96,0.03,0.000001,0.000100,30,15,11.322,0.041777,0.038889,0.014723,0.047784,0.222222,0.039610
7,bpr,96,0.08,0.000100,0.000010,10,3,3.606,0.041777,0.038889,0.010569,0.039415,0.222222,0.040758
8,warp,96,0.08,0.000010,0.000100,10,16,2.261,0.039643,0.050000,0.009124,0.042810,0.388889,0.042193
9,warp,96,0.03,0.000001,0.000010,30,2,7.490,0.039352,0.050000,0.014937,0.046240,0.222222,0.041332


## 7. Выбор лучшего конфига



In [7]:
print('Лучший конфиг по Recall@10:')
display(pd.DataFrame([best_cfg]))
print('best Recall@10 =', round(float(best_score), 6))

top10_configs = results_val.head(10).copy()
display(top10_configs)


Лучший конфиг по Recall@10:


,loss,no_components,learning_rate,item_alpha,user_alpha,epochs,run_idx,fit_time_sec,Recall@10,Precision@10,MAP@10,NDCG@10,HitRate@10,CatalogCoverage@10
0,warp,32,0.05,0.0001,0.000001,20,36,1.928,0.052315,0.044444,0.014328,0.051355,0.333333,0.03961


best Recall@10 = 0.052315


,loss,no_components,learning_rate,item_alpha,user_alpha,epochs,run_idx,fit_time_sec,Recall@10,Precision@10,MAP@10,NDCG@10,HitRate@10,CatalogCoverage@10
0,warp,32,0.05,0.000100,0.000001,20,36,1.928,0.052315,0.044444,0.014328,0.051355,0.333333,0.039610
1,bpr,96,0.03,0.000010,0.000001,30,7,11.132,0.048721,0.044444,0.017389,0.054343,0.222222,0.041619
2,warp,96,0.03,0.000100,0.000010,20,19,5.275,0.048611,0.044444,0.012002,0.043677,0.222222,0.039036
3,warp,64,0.03,0.000001,0.000100,20,32,3.874,0.048225,0.044444,0.018770,0.050249,0.222222,0.039610
4,bpr,96,0.08,0.000001,0.000010,20,9,7.317,0.043474,0.038889,0.010202,0.041415,0.277778,0.044776
5,bpr,64,0.05,0.000010,0.000001,30,4,7.654,0.042394,0.038889,0.012428,0.046729,0.277778,0.041332
6,bpr,96,0.03,0.000001,0.000100,30,15,11.322,0.041777,0.038889,0.014723,0.047784,0.222222,0.039610
7,bpr,96,0.08,0.000100,0.000010,10,3,3.606,0.041777,0.038889,0.010569,0.039415,0.222222,0.040758
8,warp,96,0.08,0.000010,0.000100,10,16,2.261,0.039643,0.050000,0.009124,0.042810,0.388889,0.042193
9,warp,96,0.03,0.000001,0.000010,30,2,7.490,0.039352,0.050000,0.014937,0.046240,0.222222,0.041332


## 8. Оценка лучшей модели на test


In [8]:
val_pred_best = build_predictions(best_model, eval_users_val, 'validation', K_MAX)
test_pred_best = build_predictions(best_model, eval_users_test, 'test', K_MAX)

val_metrics_best = evaluate_split(val_pred_best, val_gt, eval_users_val, candidate_items, K_VALUES)
val_metrics_best.insert(0, 'split', 'validation')

test_metrics_best = evaluate_split(test_pred_best, test_gt, eval_users_test, candidate_items, K_VALUES)
test_metrics_best.insert(0, 'split', 'test')

combined_metrics_best = pd.concat([val_metrics_best, test_metrics_best], ignore_index=True)

display(val_metrics_best)
display(test_metrics_best)



,split,K,Recall@K,Precision@K,MAP@K,NDCG@K,HitRate@K,Catalog Coverage@K,average_list_length,full_topk_rate,unique_recommended_items,candidate_items
0,validation,5,0.023148,0.044444,0.010237,0.044974,0.166667,0.022675,5.0,1.0,79,3484
1,validation,10,0.052315,0.044444,0.014328,0.051355,0.333333,0.039610,10.0,1.0,138,3484
2,validation,20,0.063073,0.030556,0.015441,0.051310,0.388889,0.071183,20.0,1.0,248,3484
3,validation,50,0.113698,0.025556,0.020178,0.074350,0.555556,0.144087,50.0,1.0,502,3484


,split,K,Recall@K,Precision@K,MAP@K,NDCG@K,HitRate@K,Catalog Coverage@K,average_list_length,full_topk_rate,unique_recommended_items,candidate_items
0,test,5,0.007114,0.06250,0.004187,0.060102,0.1875,0.019518,5.0,1.0,68,3484
1,test,10,0.049357,0.08125,0.011941,0.081429,0.4375,0.035878,10.0,1.0,125,3484
2,test,20,0.067930,0.07500,0.017378,0.085029,0.4375,0.061998,20.0,1.0,216,3484
3,test,50,0.099588,0.05500,0.025468,0.083144,0.5000,0.129162,50.0,1.0,450,3484


## 9. Сравнение с baseline из ноутбука 10


In [9]:
baseline_test_path = ARTIFACTS_DIR / 'metrics' / 'lightfm_item_features' / 'test_metrics.csv'
if baseline_test_path.exists():
    base = pd.read_csv(baseline_test_path)
    cmp = test_metrics_best.merge(base, on='K', suffixes=('_tuned', '_baseline'))

    compare_rows = []
    for _, r in cmp.iterrows():
        compare_rows.append({
            'K': int(r['K']),
            'Recall_delta': float(r['Recall@K_tuned'] - r['Recall@K_baseline']),
            'Precision_delta': float(r['Precision@K_tuned'] - r['Precision@K_baseline']),
            'MAP_delta': float(r['MAP@K_tuned'] - r['MAP@K_baseline']),
            'NDCG_delta': float(r['NDCG@K_tuned'] - r['NDCG@K_baseline']),
            'HitRate_delta': float(r['HitRate@K_tuned'] - r['HitRate@K_baseline']),
            'CatalogCoverage_delta': float(r['Catalog Coverage@K_tuned'] - r['Catalog Coverage@K_baseline']),
        })
    compare_df = pd.DataFrame(compare_rows)
    display(compare_df)
else:
    compare_df = None
    print('Базовые LightFM метрики не найдены, сравнение пропущено.')



,K,Recall_delta,Precision_delta,MAP_delta,NDCG_delta,HitRate_delta,CatalogCoverage_delta
0,5,-0.020080,-5.000000e-02,-0.074563,-0.046409,-0.0625,0.000861
1,10,0.015049,-1.250000e-02,-0.053317,-0.016606,0.1875,0.002296
2,20,0.014723,1.387779e-17,-0.027287,-0.003152,0.0000,0.002296
3,50,-0.033770,-7.500000e-03,-0.008805,-0.015760,-0.1250,0.002870


## 10. Сохранение результатов


In [10]:
# 1) tuning tables
val_tuning_path = TUNING_DIR / 'lightfm_tuning_validation_results.csv'
results_val.to_csv(val_tuning_path, index=False)

top10_path = TUNING_DIR / 'top10_configs.csv'
top10_configs.to_csv(top10_path, index=False)

# 2) best config
best_cfg_path = TUNING_DIR / 'best_config.json'
with best_cfg_path.open('w', encoding='utf-8') as f:
    json.dump(best_cfg, f, ensure_ascii=False, indent=2)

# 3) model factors
user_emb_path = MODELS_DIR / 'user_embeddings.npy'
item_emb_path = MODELS_DIR / 'item_embeddings.npy'
item_biases_path = MODELS_DIR / 'item_biases.npy'
user_biases_path = MODELS_DIR / 'user_biases.npy'

np.save(user_emb_path, best_model.user_embeddings)
np.save(item_emb_path, best_model.item_embeddings)
np.save(item_biases_path, best_model.item_biases)
np.save(user_biases_path, best_model.user_biases)

# 4) predictions
val_pred_path = PRED_DIR / f'val_predictions_top{K_MAX}.csv'
test_pred_path = PRED_DIR / f'test_predictions_top{K_MAX}.csv'
val_pred_best.to_csv(val_pred_path, index=False)
test_pred_best.to_csv(test_pred_path, index=False)

# 5) metrics
val_metrics_path = METRICS_DIR / 'validation_metrics.csv'
test_metrics_path = METRICS_DIR / 'test_metrics.csv'
combined_metrics_path = METRICS_DIR / 'combined_metrics.csv'

val_metrics_best.to_csv(val_metrics_path, index=False)
test_metrics_best.to_csv(test_metrics_path, index=False)

# 6) compare with baseline
compare_path = METRICS_DIR / 'compare_with_baseline_lightfm.csv'
if compare_df is not None:
    compare_df.to_csv(compare_path, index=False)

# 7) metadata
metadata = {
    'model_name': 'lightfm_item_features_tuned',
    'dataset': split_meta.get('active_dataset') if isinstance(split_meta, dict) else None,
    'objective': 'Recall@10 on validation',
    'best_config': best_cfg,
    'k_values': K_VALUES,
    'paths': {
        'val_tuning_results': str(val_tuning_path),
        'top10_configs': str(top10_path),
        'best_config': str(best_cfg_path),
        'user_embeddings': str(user_emb_path),
        'item_embeddings': str(item_emb_path),
        'item_biases': str(item_biases_path),
        'user_biases': str(user_biases_path),
        'val_predictions': str(val_pred_path),
        'test_predictions': str(test_pred_path),
        'validation_metrics': str(val_metrics_path),
        'test_metrics': str(test_metrics_path),
        'compare_with_baseline': str(compare_path),
    }
}

meta_path = MODELS_DIR / 'model_metadata.json'
with meta_path.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)


## 11. Итог
- Для двухэтапной модели best-конфиг выбирается по максимальному `Recall@10` на validation

